# Step 2b: The hide-and-predict trick, applied to the simplest baseline

Our global-mean baseline scored **RMSE 1.0591** on the full subsample.
Now we test something important: does that number hold up on ratings
the model never saw?

**The trick:** split our known ratings into two piles —
- **Visible** (80%) — the model is allowed to learn from this
- **Hidden** (20%) — we know the real answer, but we hide it from the model

We compute the global mean using ONLY the visible 80%, then "predict"
every hidden rating with that mean, then compare to the real value we
hid. If the RMSE on the hidden pile looks similar to the RMSE on
everything, that's evidence the model generalizes — which is exactly
what Kaggle's hidden test set will check for real.

In [1]:
DATA_DIR = "data"   # adjust if your folder is named differently
SUBSAMPLE_SIZE = 500_000

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

train = pd.read_csv(f"{DATA_DIR}/train.csv")
if len(train) > SUBSAMPLE_SIZE:
    train = train.sample(n=SUBSAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Working with: {train.shape}")

Working with: (500000, 4)


## Split into visible and hidden

In [3]:
visible, hidden = train_test_split(train, test_size=0.2, random_state=42)

print(f"Visible (model can see): {len(visible):,}")
print(f"Hidden  (model cannot see): {len(hidden):,}")

Visible (model can see): 400,000
Hidden  (model cannot see): 100,000


## Compute the baseline using ONLY the visible 80%

This is the key step — the global mean is calculated without ever
looking at the hidden ratings.

In [4]:
global_mean_visible_only = visible["rating"].mean()
print(f"Global mean from visible 80% only: {global_mean_visible_only:.4f}")

Global mean from visible 80% only: 3.5334


## Predict the hidden ratings and check RMSE

We apply that mean to every hidden row, then compare the prediction
to the REAL rating — which we're only allowed to look at now, for
scoring purposes.

In [5]:
hidden = hidden.copy()
hidden["predicted"] = global_mean_visible_only   # same number for every row

rmse_hidden = np.sqrt(mean_squared_error(hidden["rating"], hidden["predicted"]))
print(f"RMSE on hidden ratings: {rmse_hidden:.4f}")

RMSE on hidden ratings: 1.0577


## Compare to the no-hiding baseline

This confirms the hidden-data RMSE isn't just a fluke — it should be
very close to the RMSE we got using all the data with no split at all.

In [6]:
full_mean = train["rating"].mean()
full_pred_array = np.full(len(train), full_mean)
rmse_full = np.sqrt(mean_squared_error(train["rating"], full_pred_array))

print(f"RMSE using all data (no hiding):     {rmse_full:.4f}")
print(f"RMSE on hidden-only data (the trick): {rmse_hidden:.4f}")
print(f"Difference: {abs(rmse_full - rmse_hidden):.4f}")

RMSE using all data (no hiding):     1.0591
RMSE on hidden-only data (the trick): 1.0577
Difference: 0.0014


## What this tells us

If the two RMSE numbers are close (a difference of a few thousandths,
not tenths), that's the proof-of-concept: even this trivial model's
performance on *unseen* data matches its performance on data it was
computed from. There's no overfitting happening — there's barely
anything TO overfit, since the model is just one number.

This matters because it validates the METHOD, not just the model.
We'll reuse this exact hide-then-predict pattern for every future
model — user/movie bias, SVD++, LightGBM — and a similar small gap
between "visible" and "hidden" RMSE will be our signal that each
model is generalizing rather than memorizing.

## Next step

Build a smarter baseline: predict using user bias + movie bias
instead of just the global mean, and re-run this same hide-and-predict
check to see the RMSE improve.